# GoEmotions — Exploratory Data Analysis

This notebook explores the GoEmotions dataset (28-class, multi-label emotion classification).

**Sections:**
1. Imports & setup
2. Load dataset
3. Label distribution plot
4. Multi-label statistics
5. Text length distribution
6. Sample examples per emotion (top 5)
7. Key insights summary

## 1. Imports & Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Make src importable from the notebook
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import EMOTION_NAMES, NUM_LABELS

# Output directory
EDA_DIR = ROOT / "results" / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
print(f"Root: {ROOT}")
print(f"EDA output: {EDA_DIR}")

## 2. Load GoEmotions Dataset

In [ ]:
from src.data import load_goemotions

ds = load_goemotions(config="simplified")
print(ds)

train_ds = ds["train"]
val_ds   = ds["validation"]
test_ds  = ds["test"]

print(f"\nTrain : {len(train_ds):,} samples")
print(f"Val   : {len(val_ds):,} samples")
print(f"Test  : {len(test_ds):,} samples")
print(f"Total : {len(train_ds)+len(val_ds)+len(test_ds):,} samples")

## 3. Label Distribution Plot

In [ ]:
# Count occurrences of each label in the TRAIN split
label_counts = np.zeros(NUM_LABELS, dtype=int)
for item in train_ds:
    for idx in item["labels"]:
        label_counts[idx] += 1

# Sort descending
sorted_idx = np.argsort(label_counts)[::-1]
sorted_names  = [EMOTION_NAMES[i] for i in sorted_idx]
sorted_counts = label_counts[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(sorted_names[::-1], sorted_counts[::-1],
               color=sns.color_palette("Blues_d", NUM_LABELS))

for bar, cnt in zip(bars, sorted_counts[::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
            f"{cnt:,}", va="center", ha="left", fontsize=8)

ax.set_xlabel("Number of training samples", fontsize=12)
ax.set_title("GoEmotions — Label Distribution (train split)", fontsize=14, pad=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(EDA_DIR / "label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {EDA_DIR / 'label_distribution.png'}")

## 4. Multi-label Statistics

In [ ]:
from collections import Counter

# --- label cardinality (how many labels per sample?) ---
cardinality = [len(item["labels"]) for item in train_ds]
card_counter = Counter(cardinality)

print("=== Label Cardinality (train) ===")
for k in sorted(card_counter):
    pct = 100 * card_counter[k] / len(train_ds)
    print(f"  {k} label(s): {card_counter[k]:>6,}  ({pct:.1f}%)")

multi_label_pct = 100 * sum(v for k, v in card_counter.items() if k >= 2) / len(train_ds)
print(f"\nMulti-label samples: {multi_label_pct:.1f}%")

# --- imbalance ratio ---
max_count = label_counts.max()
min_count = label_counts[label_counts > 0].min()
print(f"\nClass imbalance ratio (max/min): {max_count / min_count:.0f}×")
print(f"  Most common : {EMOTION_NAMES[label_counts.argmax()]}  ({max_count:,})")
print(f"  Least common: {EMOTION_NAMES[label_counts.argmin()]}  ({min_count:,})")

# --- Plot cardinality ---
fig, ax = plt.subplots(figsize=(7, 4))
max_card = max(card_counter)
xs = list(range(1, max_card + 1))
ys = [card_counter.get(x, 0) for x in xs]
ax.bar(xs, ys, color="steelblue", edgecolor="white")
ax.set_xlabel("Number of labels per sample")
ax.set_ylabel("Count")
ax.set_title("Label Cardinality Distribution (train split)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(EDA_DIR / "label_cardinality.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Text Length Distribution

In [ ]:
# Word-level token count (simple whitespace split)
lengths = [len(item["text"].split()) for item in train_ds]
lengths = np.array(lengths)

print("=== Text Length Statistics (word count, train split) ===")
print(f"  Min    : {lengths.min()}")
print(f"  Median : {np.median(lengths):.0f}")
print(f"  Mean   : {lengths.mean():.1f}")
print(f"  P95    : {np.percentile(lengths, 95):.0f}")
print(f"  Max    : {lengths.max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(lengths, bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(np.median(lengths), color="orange", linestyle="--", label=f"Median={np.median(lengths):.0f}")
axes[0].set_xlabel("Word count")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Text Length Distribution")
axes[0].legend()

# Zoomed (≤80 words covers 99% of data)
axes[1].hist(lengths[lengths <= 80], bins=40, color="teal", edgecolor="white")
axes[1].axvline(128, color="red", linestyle="--", label="Tokenizer max_len=128")
axes[1].set_xlabel("Word count (≤80)")
axes[1].set_title("Zoomed View (≤80 words)")
axes[1].legend()

plt.suptitle("GoEmotions — Text Length (train split)", fontsize=13)
plt.tight_layout()
plt.savefig(EDA_DIR / "text_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

truncated = (lengths > 128).sum()
print(f"\nSamples with >128 words (would be truncated): {truncated} ({100*truncated/len(lengths):.1f}%)")

## 6. Sample Examples per Emotion (top 5 most common)

In [ ]:
# Collect examples: for top-5 emotions, show 3 sample texts
top5_indices = sorted_idx[:5]

examples_by_emotion = {EMOTION_NAMES[i]: [] for i in top5_indices}

for item in train_ds:
    for idx in item["labels"]:
        name = EMOTION_NAMES[idx]
        if name in examples_by_emotion and len(examples_by_emotion[name]) < 3:
            examples_by_emotion[name].append(item["text"])

    # Stop early once we have enough examples
    if all(len(v) >= 3 for v in examples_by_emotion.values()):
        break

print("=== Sample Examples for Top-5 Most Common Emotions ===\n")
for emotion, texts in examples_by_emotion.items():
    count = label_counts[EMOTION_NAMES.index(emotion)]
    print(f"── {emotion.upper()}  (train count: {count:,}) ──")
    for i, t in enumerate(texts, 1):
        print(f"  {i}. {t[:120]}")
    print()

## 7. Key Insights Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS — GoEmotions EDA")
print("="*60)

print(f"""
1. Dataset size
   Train {len(train_ds):,} / Val {len(val_ds):,} / Test {len(test_ds):,}
   Total: {len(train_ds)+len(val_ds)+len(test_ds):,} Reddit comments

2. Multi-label
   {multi_label_pct:.1f}% of training samples have ≥2 emotion labels.
   → Binary cross-entropy (not softmax) is required.

3. Class imbalance
   Most common : '{EMOTION_NAMES[label_counts.argmax()]}' ({label_counts.max():,} samples)
   Least common: '{EMOTION_NAMES[label_counts.argmin()]}' ({label_counts.min():,} samples)
   Ratio        : {label_counts.max() / label_counts[label_counts>0].min():.0f}×
   → Use pos_weight in BCEWithLogitsLoss; use F1-macro as primary metric.

4. Text length
   Median {np.median(lengths):.0f} words; 95th-percentile {np.percentile(lengths, 95):.0f} words.
   Only {truncated} samples ({100*truncated/len(lengths):.1f}%) exceed the 128-token limit.
   → max_length=128 is a safe choice.

5. Modelling implications
   - BERT/RoBERTa fine-tuning: BCEWithLogitsLoss + pos_weight, 3 epochs.
   - LLM prompting: must instruct multi-label output; include neutral fallback.
   - Primary metric: F1-macro (equal weight to rare classes).
""")